# 04 — Federated FedAvg baseline (Phase 3)

Simulated hospitals (Dirichlet ward-mixture) train together via **FedAvg** (Flower).
Compares **pooled** (Phase 1) vs **local-only** (each hospital alone) vs **FedAvg**.

**To get new code after a `git pull`:** just re-run cell 1 (pull) then the run cell —
cells 4/5 force-reload `amr_fed` from disk, so **no kernel restart is needed**.
All simulated on ONE machine — no second computer needed.

In [ ]:
# 1) Get the code + deps
!git clone -b phase1-core-pipeline https://github.com/RawEgg6/Capstone-amr-fed.git 2>/dev/null || (cd Capstone-amr-fed && git fetch && git checkout phase1-core-pipeline && git pull)
!pip install -q torch_geometric 'flwr[simulation]'

In [ ]:
# 2) Point at the data (mount Drive, set ARMD_DIR before importing amr_fed)
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['ARMD_DIR'] = '/content/drive/MyDrive/ARMD'   # EDIT to your ARMD folder

In [ ]:
# 3) Sanity: data resolves
import sys
sys.path.insert(0, '/content/Capstone-amr-fed/src')
from amr_fed import config
from pathlib import Path
D = Path(config.DATA_DIR)
print('DATA_DIR:', D, '| exists:', D.exists())
assert D.exists(), 'ARMD_DIR is wrong — fix cell 2 and re-run.'

In [ ]:
# 4) Run FedAvg at alpha=0.5 (5 hospitals). Prints local-only vs FedAvg vs pooled.
# Force-reload amr_fed from disk so a `git pull` takes effect WITHOUT a kernel restart.
import sys
for _m in [m for m in list(sys.modules) if m.startswith('amr_fed')]:
    del sys.modules[_m]

from amr_fed.federated.run import run_fedavg
res = run_fedavg(alpha=0.5, n_clients=5, rounds=10, local_epochs=6)
print(res)

In [ ]:
# 5) (optional) sweep heterogeneity — does the federated gain grow as hospitals diverge?
import sys
for _m in [m for m in list(sys.modules) if m.startswith('amr_fed')]:
    del sys.modules[_m]
from amr_fed.federated.run import run_fedavg
for a in (0.1, 0.5, 1.0):
    run_fedavg(alpha=a, n_clients=5, rounds=10, local_epochs=6)